In [ ]:
import numpy as np
from qiskit.quantum_info import Operator
from scipy.linalg import inv
from qiskit import QuantumCircuit
import math
from qiskit import QuantumRegister, ClassicalRegister

def build_system_matrix_L(m, k, p, A, h):
    """
    Constructs the linear system matrix L = I - N for the quantum 
    ODE/DAE solver algorithm based on the provided block equations.
    """
    d = A.shape[0]  # Dimension of the system matrix A
    
    # 1. Determine the dimensions of the quantum registers
    dim_j = k + 1          # j goes from 0 to k, so size is k + 1
    dim_i = m + p + 1      # max index is i+1 = m+p, so size is m + p + 1
    
    I_sys = np.eye(d)
    
    # ==========================================
    # Equation 3: Construct M1
    # ==========================================
    M1 = np.zeros((dim_j * d, dim_j * d), dtype=complex)
    for j in range(k):
        # Create |j+1><j|
        shift_j = np.zeros((dim_j, dim_j))
        shift_j[j+1, j] = 1
        
        # Tensor product with (Ah) / (j+1)
        term = (A * h) / (j + 1)
        M1 += np.kron(shift_j, term)
        
    # ==========================================
    # Equation 4: Construct M2
    # ==========================================
    M2 = np.zeros((dim_j * d, dim_j * d), dtype=complex)
    for j in range(k + 1):
        # Create |0><j|
        proj_j = np.zeros((dim_j, dim_j))
        proj_j[0, j] = 1
        
        # Tensor product with Identity
        M2 += np.kron(proj_j, I_sys)
        
    # ==========================================
    # Equation 2: Construct N
    # ==========================================
    # Calculate M2 * (I - M1)^-1
    I_M1 = np.eye(dim_j * d) - M1
    I_M1_inv = inv(I_M1)
    block_1 = M2 @ I_M1_inv
    
    I_j_sys = np.eye(dim_j * d)
    N = np.zeros((dim_i * dim_j * d, dim_i * dim_j * d), dtype=complex)
    
    # First summation term (i = 0 to m)
    for i in range(m + 1):
        shift_i = np.zeros((dim_i, dim_i))
        shift_i[i+1, i] = 1
        N += np.kron(shift_i, block_1)
        
    # Second summation term (i = m+1 to m+p-1)
    for i in range(m + 1, m + p):
        shift_i = np.zeros((dim_i, dim_i))
        shift_i[i+1, i] = 1
        N += np.kron(shift_i, I_j_sys)
        
    # ==========================================
    # Equation 1: Construct L = I - N
    # ==========================================
    I_total = np.eye(dim_i * dim_j * d)
    L_matrix = I_total - N
    
    return L_matrix

# ==========================================
# Example Usage
# ==========================================
# Provide the parameters for your simulation
m_val = 5  # Time steps
k_val = 3  # Truncation order
p_val = 2  # Delay/padding steps
h_val = 0.1 # Time step size

# Provide the physical system matrix A
# Using a generic 2x2 matrix as a placeholder
A_matrix = np.array([[1, 1], 
                     [1, -1]]) # Example system matrix (normalized)

# Generate the raw numpy matrix
L = build_system_matrix_L(m_val, k_val, p_val, A_matrix, h_val)
L_op = Operator(L)


print((m_val+p_val+1)*(k_val+1)*A_matrix.shape[0])

dim_j = k_val + 1
dim_i = m_val + p_val + 1
d = A_matrix.shape[0]

num_i = max(1, int(math.ceil(math.log2(dim_i))))
num_j = max(1, int(math.ceil(math.log2(dim_j))))
num_sys = max(1, int(math.ceil(math.log2(d))))

ancilla = QuantumRegister(1, 'ancilla_lcu')
reg_i = QuantumRegister(num_i, 'time_i')
reg_j = QuantumRegister(num_j, 'taylor_j')
reg_sys = QuantumRegister(num_sys, 'system_d')
cr = ClassicalRegister(1, 'post_select')

qc = QuantumCircuit(ancilla, reg_i, reg_j, reg_sys, cr)
qc.append(L_op, reg_i[:] + reg_j[:] + reg_sys[:])




qc.draw("mpl")


64


ValueError: Input matrix is not unitary.